## AMCL Performance Analysis

In [1]:
import math
import numpy as np
import pandas as pd
from rosbags.rosbag2 import Reader

"""
Analysis Outline
Collected Topics
- /amcl_pose, /particle_cloud, /odometry/filtered
Scenerio
- took 2 turns around 3 waypoints, stopping for 5 seconds at each waypoint after the waypoint is reached

Waypoint coordinates relative to the map frame in detail

w1->(1.18, 2.52, pi/2)
w2->(-0.75, 2.57, pi/2)
w3->(0,0,pi/2)

Tasks
->Capture the arrival times for all the 6 points
->store /amcl_pose + /odometry/filtered estimates for all the points / (x,y,theta).
->calculate statistics: mean absolute error, mean_squared_error, estimates
"""

GROUND_TRUTHS = {
    "w1": (1.18, 2.52, math.pi/2),
    "w2": (-0.75, 2.57, math.pi/2),
    "w3": ((0, 0, math.pi/2))
}

# --- CONFIGURATION ---
BAG_PATH = "/home/rosgeek/Documents/lab_parkur_ws/src/lab_mapping/rosbags/rosbag2_2026_04_20-07_28_15"  # Folder containing the .db3
TOPIC = "/amcl_pose"

In [2]:
from pathlib import Path
from rosbags.highlevel import AnyReader
from rosbags.typesys import Stores, get_typestore
import datetime

class Pose:
     @classmethod
     def get_average(self, poses):
          n = len(poses)
          sum_pose = Pose()
          for pose in poses:
               sum_pose.x+=pose.x
               sum_pose.y+=pose.y
               sum_pose.theta+=pose.theta
          sum_pose.x /= n
          sum_pose.y /= n
          sum_pose.theta /= n

          return sum_pose

     def __init__(self, x=0.0,y=0.0,theta=0.0,time=0.0):
          self.x = x
          self.y = y
          self.theta = theta
          self.time = time
          
     def __repr__(self):
          return f"x: {self.x}, y: {self.y}, theta: {self.theta}, time: {self.time}"
     
def quaternion_to_yaw(qx, qy, qz, qw):
    """Converts 2D planar quaternion to Euler Yaw (theta) in radians."""
    siny_cosp = 2 * (qw * qz + qx * qy)
    cosy_cosp = 1 - 2 * (qy * qy + qz * qz)
    return math.atan2(siny_cosp, cosy_cosp)

def timestamp_to_hour_minute(timestamp):
     return datetime.datetime.fromtimestamp(timestamp).strftime('%H:%M:%S')

In [3]:
bagpath = Path(BAG_PATH)
typestore = get_typestore(Stores.ROS2_FOXY)

amcl_poses = [] #Pose array
odom_filtered_speeds = []


# Create reader instance and open for reading.
with AnyReader([bagpath], default_typestore=typestore) as reader:
    connections = [x for x in reader.connections if x.topic == '/amcl_pose']
    for connection, timestamp, rawdata in reader.messages(connections=connections):
         msg = reader.deserialize(rawdata, connection.msgtype)
         yaw = quaternion_to_yaw(msg.pose.pose.orientation.x, 
                                 msg.pose.pose.orientation.y, 
                                 msg.pose.pose.orientation.z, 
                                 msg.pose.pose.orientation.w)
         x,y = msg.pose.pose.position.x, msg.pose.pose.position.y
         time = msg.header.stamp.sec + msg.header.stamp.nanosec / 1e9
         pose = Pose(x,y,yaw,time)
         amcl_poses.append(pose)

    connections = [x for x in reader.connections if x.topic == '/odometry/filtered']
    for connection, timestamp, rawdata in reader.messages(connections=connections):
         msg = reader.deserialize(rawdata, connection.msgtype)
         vx, vy = msg.twist.twist.linear.x, msg.twist.twist.linear.y
         v = np.sqrt(vx**2 + vy**2)

         time = msg.header.stamp.sec + msg.header.stamp.nanosec / 1e9
         odom_filtered_speeds.append((v, time))

for speed, time in odom_filtered_speeds:
     print(f"speed: {speed}, time: {time}")

speed: 0.0, time: 1776670095.416504
speed: 0.0, time: 1776670095.4562383
speed: 0.0, time: 1776670095.4961183
speed: 0.0, time: 1776670095.5166583
speed: 0.0, time: 1776670095.5563288
speed: 0.0, time: 1776670095.5962806
speed: 0.0, time: 1776670095.61679
speed: 0.0, time: 1776670095.656255
speed: 0.0, time: 1776670095.69635
speed: 0.0, time: 1776670095.7167654
speed: 0.0, time: 1776670095.7564447
speed: 0.0, time: 1776670095.7966084
speed: 0.0, time: 1776670095.8167582
speed: 0.0, time: 1776670095.935485
speed: 0.0, time: 1776670095.9961321
speed: 0.0, time: 1776670096.0167487
speed: 0.0, time: 1776670096.0565085
speed: 0.0, time: 1776670096.0962048
speed: 0.0, time: 1776670096.116817
speed: 0.0, time: 1776670096.1566558
speed: 0.0, time: 1776670096.1965048
speed: 0.0, time: 1776670096.2168922
speed: 0.0, time: 1776670096.2563586
speed: 0.0, time: 1776670096.2963004
speed: 0.0, time: 1776670096.316751
speed: 0.0, time: 1776670096.356546
speed: 0.0, time: 1776670096.3965282
speed: 0.0,

In [4]:
# Take moving average of the odom_filtered measurements
moving_sum = 0
window = round(6 / 0.04) # Moving average over 6 seconds
speed_averages = []

for i in range(len(odom_filtered_speeds)):
    if i % window != 0:
        moving_sum += odom_filtered_speeds[i][0] 
    else:
        avg_speed = moving_sum / window
        time = timestamp_to_hour_minute(round(odom_filtered_speeds[i][1]))
        speed_averages.append((avg_speed, time))
        moving_sum = 0


In [5]:
for avg, time in speed_averages:
    print(f"avg: {avg}, time: {time}")

avg: 0.0, time: 10:28:15
avg: 0.0, time: 10:28:21
avg: 0.0, time: 10:28:27
avg: 0.0, time: 10:28:32
avg: 0.0, time: 10:28:38
avg: 0.0, time: 10:28:43
avg: 0.0, time: 10:28:49
avg: 0.0, time: 10:28:54
avg: 0.29634645470949034, time: 10:29:00
avg: 0.40677175233886764, time: 10:29:06
avg: 0.13624608557722387, time: 10:29:11
avg: 0.1288112842995242, time: 10:29:17
avg: 0.016380487602539552, time: 10:29:22
avg: 0.00021304917175351084, time: 10:29:28
avg: 6.086914044113476e-07, time: 10:29:34
avg: 5.812669056167498e-09, time: 10:29:39
avg: 0.23026628741894167, time: 10:29:45
avg: 0.1887507447794904, time: 10:29:50
avg: 0.417993995110887, time: 10:29:56
avg: 0.2804442290961021, time: 10:30:01
avg: 0.24773894912996866, time: 10:30:07
avg: 0.0008891219268879338, time: 10:30:13
avg: 0.0001776573452219359, time: 10:30:18
avg: 2.6304475779197194e-07, time: 10:30:24
avg: 0.0439159994938971, time: 10:30:29
avg: 0.5005513100867263, time: 10:30:35
avg: 0.1693774870261084, time: 10:30:40
avg: 0.1920821

### Stop Times Using the Simulation Data
- simulation start time: 
- w1.1->avg: 6.086914044113476e-07, time: 10:29:34, 1776670174
- w2.1->avg: 0.0001776573452219359, time: 10:30:18, 1776670218
- w3.1->avg: 0.00013190480794947367, time: 10:30:57, 1776670257
- w1.2->avg: 0.00019086416096986196, time: 10:31:31, 1776670291
- w2.2->avg: 0.0002274635274270631, time: 10:32:10, 1776670330
- w3.2->avg: 3.316996497355907e-13, time: 10:32:54,  1776670374

- w1_1->43 seconds
- w2_1->87 seconds
- w3_1->158 seconds
- w1_2->190 seconds
- w2_2->250 seconds
- w3_3->316 seconds

->AMCL pose estimate interval->take the estimate closest to the given second + take the error as pose difference

In [20]:
for pose in amcl_poses:
    print(pose.__repr__())

x: 0.16171772778594706, y: 1.486880491648654, theta: 2.351795913202608, time: 1776362370.6748762
x: 0.1561879383336947, y: 1.472511020301647, theta: 1.8646216224284176, time: 1776362414.8935726
x: 0.16266854318482266, y: 1.4461693306626948, theta: 1.6481296827787604, time: 1776362415.2348773
x: 0.1138463709945193, y: 1.4342846251387498, theta: 1.2353258657697315, time: 1776362415.6593704
x: 0.11065546137593323, y: 1.4175438228296176, theta: 0.9475962844557544, time: 1776362416.0837913
x: 0.09151808787631247, y: 1.4086264548005714, theta: 0.7114152231213682, time: 1776362416.5104337
x: 0.09923378601921287, y: 1.422904248300159, theta: 0.4994093497338695, time: 1776362416.9308891
x: 0.14166991508025276, y: 1.4289694975286498, theta: 0.4989970979884598, time: 1776362417.4428735
x: 0.20781688996506137, y: 1.4461620278599607, theta: 0.6839188943426638, time: 1776362417.8591723
x: 0.2895907244152624, y: 1.4666639171750153, theta: 0.8701482190573958, time: 1776362418.2834644
x: 0.393232367791

## AMCL Poses + Differences


In [19]:
waypoint_times = {
    "w1_1": 1776362436,
    "w1_2": 1776362480,
    "w2_1": 1776362551,
    "w2_2": 1776362583,
    "w3_1": 1776362643,
    "w3_2": 1776362709
}

waypoint_poses = {
    wp: [] for wp in waypoint_times.keys()
}

#Filter amcl_pose estimations at waypoint timestamps, take the averages at each point
for wp, time in waypoint_times.items():
    if wp == "w3_2":
        waypoint_poses[wp] = Pose.get_average(amcl_poses[-3:])
    else:
        wp_estimates = [pose for pose in amcl_poses if abs(pose.time - time) < 6]
        waypoint_poses[wp] = Pose.get_average(wp_estimates)


for wp, estimate in waypoint_poses.items():
    print(f"{wp}: {estimate.__repr__()}")

w1_1: x: 1.080151440952525, y: 2.084190517926266, theta: 1.8582529817949123, time: 0.0
w1_2: x: -0.7678405053634474, y: 2.0457023861822434, theta: 1.7066128799398648, time: 0.0
w2_1: x: 0.037194002005524494, y: 0.20090182704474313, theta: 1.213305958659528, time: 0.0
w2_2: x: 0.9857595277789416, y: 2.322261348010884, theta: 1.5950750927520247, time: 0.0
w3_1: x: -0.7052279926514367, y: 2.165511568212042, theta: 1.719127643529203, time: 0.0
w3_2: x: -0.24279539292948418, y: 0.0021686803543640534, theta: 1.5226414797598042, time: 0.0


In [ ]:
GROUND_TRUTHS = {
    "w1": (1.18, 2.52, math.pi/2),
    "w2": (-0.75, 2.57, math.pi/2),
    "w3": ((0, 0, math.pi/2))
}

0.8972036348145294

## Numpy Manipulation of the Metrics

In [108]:
import numpy as np

# 1. Initialize the measurements and ground truths as numpy arrays
# Structure: [x, y, theta]
estimates = np.array([
    [1.2423772495395509, 1.99602361397758, 2.156674626743815],   # w1_1
    [1.334406073220486, 1.71118467168039, 2.313141074244669],    # w1_2
    [-0.7740814168162646, 2.531839601201153, 0.7918185778173373],# w2_1
    [-0.7765898050802513, 2.2579026014654118, 1.4806281983616432],# w2_2
    [-0.01351125246836524, 0.28308392629167767, 1.1944621025276025],# w3_1
    [-0.11539721358062677, 0.08255904437247144, 1.5801215224142322] # w3_2
])

ground_truths = np.array([
    [1.18, 2.52, 1.5707963],  # w1_1
    [1.18, 2.52, 1.5707963],  # w1_2
    [-0.75, 2.57, 1.5707963], # w2_1
    [-0.75, 2.57, 1.5707963], # w2_2
    [0.0, 0.0, 1.5707963],    # w3_1
    [0.0, 0.0, 1.5707963]     # w3_2
])

labels = ["w1_1", "w1_2", "w2_1", "w2_2", "w3_1", "w3_2"]

# 2. Calculate Individual Absolute Errors
# Separate x, y from theta for proper angular wrapping
diff_xy = estimates[:, :2] - ground_truths[:, :2]
abs_err_xy = np.abs(diff_xy)

raw_diff_theta = estimates[:, 2] - ground_truths[:, 2]
abs_err_theta = np.abs(np.arctan2(np.sin(raw_diff_theta), np.cos(raw_diff_theta)))

# Recombine into a single array of shape (6, 3)
individual_abs_errors = np.column_stack((abs_err_xy, abs_err_theta))

# 3. Calculate Means across all waypoints (axis=0 calculates the mean of each column)
mean_amcl_estimate = np.mean(estimates, axis=0)
mean_ground_truth = np.mean(ground_truths, axis=0)

# 4. Calculate Mean Absolute Error (MAE)
mean_absolute_error = np.mean(individual_abs_errors, axis=0)

# 5. Calculate Mean Squared Error (MSE)
# Square the individual absolute errors, then find the mean
mean_squared_error = np.mean(individual_abs_errors**2, axis=0)

# --- Format and Print the Results ---
print("--- INDIVIDUAL ABSOLUTE ERRORS ---")
print(f"{'Point':<6} | {'X Error (m)':<12} | {'Y Error (m)':<12} | {'Theta Error (rad)':<15}")
print("-" * 55)
for i, label in enumerate(labels):
    print(f"{label:<6} | {individual_abs_errors[i, 0]:<12.6f} | {individual_abs_errors[i, 1]:<12.6f} | {individual_abs_errors[i, 2]:<15.6f}")

print("\n--- AGGREGATE METRICS ---")
print(f"{'Metric':<25} | {'X':<12} | {'Y':<12} | {'Theta':<12}")
print("-" * 60)
print(f"{'Mean AMCL Estimate':<25} | {mean_amcl_estimate[0]:<12.6f} | {mean_amcl_estimate[1]:<12.6f} | {mean_amcl_estimate[2]:<12.6f}")
print(f"{'Mean Ground Truth':<25} | {mean_ground_truth[0]:<12.6f} | {mean_ground_truth[1]:<12.6f} | {mean_ground_truth[2]:<12.6f}")
print(f"{'Mean Absolute Error (MAE)':<25} | {mean_absolute_error[0]:<12.6f} | {mean_absolute_error[1]:<12.6f} | {mean_absolute_error[2]:<12.6f}")
print(f"{'Mean Squared Error (MSE)':<25} | {mean_squared_error[0]:<12.6f} | {mean_squared_error[1]:<12.6f} | {mean_squared_error[2]:<12.6f}")

--- INDIVIDUAL ABSOLUTE ERRORS ---
Point  | X Error (m)  | Y Error (m)  | Theta Error (rad)
-------------------------------------------------------
w1_1   | 0.062377     | 0.523976     | 0.585878       
w1_2   | 0.154406     | 0.808815     | 0.742345       
w2_1   | 0.024081     | 0.038160     | 0.778978       
w2_2   | 0.026590     | 0.312097     | 0.090168       
w3_1   | 0.013511     | 0.283084     | 0.376334       
w3_2   | 0.115397     | 0.082559     | 0.009325       

--- AGGREGATE METRICS ---
Metric                    | X            | Y            | Theta       
------------------------------------------------------------
Mean AMCL Estimate        | 0.149534     | 1.477099     | 1.586141    
Mean Ground Truth         | 0.143333     | 1.696667     | 1.570796    
Mean Absolute Error (MAE) | 0.066061     | 0.341449     | 0.430505    
Mean Squared Error (MSE)  | 0.007086     | 0.185758     | 0.275163    
